# Lecture 02 — HTTP in Python

> *"99% of crawler bugs are HTTP bugs in disguise."*

You already wrote your first `httpx.get()` in Lecture 00. This lecture covers everything *around* that one line: headers, sessions, cookies, status codes, redirects, timeouts, retries. By the end you'll be able to fetch any public URL the way a real crawler does — politely, robustly, and identifiable.

## What you'll be able to do after this lecture

- Choose between `httpx` and `requests`, and explain the tradeoff.
- Set headers (especially `User-Agent`) on every request.
- Use a `Client`/`Session` object to share cookies and connections.
- Read status codes and react correctly to each class.
- Configure timeouts and write a retry loop with exponential backoff.
- Handle redirects, query parameters, and POST forms.

## Setup

```bash
pip install httpx beautifulsoup4 lxml tenacity
```


## 1. `requests` vs `httpx`

You'll see both in the wild. They're nearly identical, intentionally. The decision tree:

| Need                         | Pick           |
|------------------------------|----------------|
| Beginner-friendly, sync only | `requests`     |
| Async (`async`/`await`)      | `httpx`        |
| HTTP/2 support               | `httpx`        |
| Smaller dependency footprint | `httpx` ≈ `requests` |
| Bigger ecosystem of plugins  | `requests`     |

We use **`httpx`** in this course because we'll go async in Lecture 06 and we want one library for both. Anything we show with `httpx` translates one-for-one to `requests` (rename `httpx.Client` → `requests.Session` and most code keeps working).

In [ ]:
import httpx

r = httpx.get("https://httpbin.org/get")
print(r.status_code)
print(r.json())


`httpbin.org` is a free service that echoes back whatever you send. It's the perfect playground for HTTP experiments — we'll use it heavily in this lecture.

## 2. Headers, and why they matter

Every request carries headers. Most servers don't care, but the ones that do can be picky.

The two headers you'll set most often:

- **`User-Agent`** — identifies your client. Default Python `User-Agent` is recognizably non-browser and many sites block it.
- **`Accept-Language`** — some servers serve different content based on this. Worth setting if you want consistent results.


In [ ]:
# Send a request and let httpbin echo back what we sent
r = httpx.get("https://httpbin.org/headers")
print(r.json()["headers"])


In [ ]:
# Now with custom headers — this is what your real crawler should look like
headers = {
    "User-Agent": "CrawlingTutorial/0.1 (+https://github.com/Vladimir-125/CrawlingTutorial; learning)",
    "Accept-Language": "en-US,en;q=0.9",
}

r = httpx.get("https://httpbin.org/headers", headers=headers)
print(r.json()["headers"])


### Should I pretend to be Chrome?

Some tutorials tell you to copy a real browser's `User-Agent` string verbatim. That's a tradeoff:

- **Pro**: many sites that block obvious bots will let a Chrome-shaped request through.
- **Con**: you're misrepresenting yourself. If you cause problems, the operator can't tell who you are or ask you to stop.

For learning and personal projects, the polite-identifying UA is the right default. We'll discuss the cat-and-mouse game in Lecture 08.

## 3. The `Client` (a.k.a. `Session`) — use it always

Every `httpx.get(...)` call opens a fresh TCP connection, performs a TLS handshake, and tears it down. For a crawler that fetches hundreds of pages from the same host, that's pure waste.

A **`Client`** keeps connections open, shares cookies across requests, and lets you set defaults (headers, timeouts) once. This is the single biggest performance + correctness improvement in your toolkit. Use it always.

In [ ]:
DEFAULT_HEADERS = {
    "User-Agent": "CrawlingTutorial/0.1 (+contact@example.com)",
    "Accept-Language": "en-US,en;q=0.9",
}

with httpx.Client(headers=DEFAULT_HEADERS, timeout=10.0, follow_redirects=True) as client:
    r1 = client.get("https://httpbin.org/cookies/set?session=abc123")
    r2 = client.get("https://httpbin.org/cookies")
    print("After login:", r2.json())


Notice that the cookie set on the first request is automatically sent on the second. That's the `Client` carrying state. Without it you'd have to do that bookkeeping yourself.

(`requests` users: the equivalent is `with requests.Session() as s: ...`. Same idea.)

## 4. Status codes: a field guide

| Code | Name                  | What to do                                                |
|-----:|-----------------------|-----------------------------------------------------------|
| 200  | OK                    | Parse the body.                                           |
| 204  | No Content            | The request worked, there's just nothing to return.       |
| 301  | Moved Permanently     | Update your seed URL to the new one.                      |
| 302  | Found                 | Follow it (default), or override if you have a reason.    |
| 304  | Not Modified          | Your cached copy is still fresh.                          |
| 400  | Bad Request           | You sent malformed data. Fix your code.                   |
| 401  | Unauthorized          | You need to log in.                                       |
| 403  | Forbidden             | You're logged in / unauthenticated, but not allowed.      |
| 404  | Not Found             | The page is gone or never existed. Don't retry.           |
| 410  | Gone                  | Permanently gone. Definitely don't retry.                 |
| 429  | Too Many Requests     | You're rate-limited. Slow down. **Always retry, with delay.** |
| 500  | Internal Server Error | Server bug. Retry, but don't hammer.                      |
| 502  | Bad Gateway           | Upstream is sick. Retry with backoff.                     |
| 503  | Service Unavailable   | Server overwhelmed or in maintenance. Retry with backoff. |
| 504  | Gateway Timeout       | Upstream didn't respond. Retry with backoff.              |

The cardinal sin is the silent failure: a 403 that gets parsed as if it were 200, leaving you with an empty result list and no error. Always check `response.status_code` (or call `response.raise_for_status()` to make it a Python exception).


In [ ]:
r = httpx.get("https://httpbin.org/status/500")
print("status:", r.status_code)

# Raise an exception on 4xx/5xx — useful inside try/except retry loops
try:
    r.raise_for_status()
except httpx.HTTPStatusError as e:
    print("caught:", type(e).__name__, e.response.status_code)


## 5. Timeouts — set them. Always.

The default `httpx` timeout is 5 seconds. The default for `requests` is *no timeout at all*. A request with no timeout will hang forever if the server stops responding mid-stream. That will hang your crawler.

`httpx` has four timeout components: `connect`, `read`, `write`, `pool`. You can set them individually or set one number for all four.

In [ ]:
# Cheap and clear: 10 seconds total per request
client = httpx.Client(timeout=10.0)

# Or fine-grained, when you have reason
client = httpx.Client(timeout=httpx.Timeout(connect=5.0, read=20.0, write=5.0, pool=5.0))


## 6. Retries with exponential backoff

Network blips happen. So do 503s. A crawler that gives up on the first failure will miss data; a crawler that retries instantly will hammer the server. The fix is **exponential backoff with jitter**.

The pattern: wait `base * 2^attempt` seconds, plus a small random jitter to spread out retries from many clients. Cap the total attempts. Only retry on transient errors (network errors, 429, 5xx). Don't retry on 404 or 403 — those won't get better.

In [ ]:
import time
import random
import httpx

TRANSIENT_STATUS = {429, 500, 502, 503, 504}

def fetch_with_retry(client: httpx.Client, url: str, max_attempts: int = 5) -> httpx.Response:
    last_error = None
    for attempt in range(max_attempts):
        try:
            r = client.get(url)
            if r.status_code in TRANSIENT_STATUS:
                raise httpx.HTTPStatusError(
                    f"transient {r.status_code}", request=r.request, response=r
                )
            return r
        except (httpx.RequestError, httpx.HTTPStatusError) as e:
            last_error = e
            if attempt == max_attempts - 1:
                break
            sleep_for = (2 ** attempt) + random.uniform(0, 1)
            print(f"  attempt {attempt+1} failed ({e}); sleeping {sleep_for:.1f}s")
            time.sleep(sleep_for)
    raise last_error

# Try it against a flaky-on-purpose endpoint
with httpx.Client(timeout=10.0, headers={"User-Agent": "CrawlingTutorial/0.1"}) as client:
    r = fetch_with_retry(client, "https://httpbin.org/status/200")
    print("final:", r.status_code)


For real projects, use the `tenacity` library — it expresses the same pattern declaratively and handles edge cases you'd otherwise have to reinvent.

In [ ]:
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

@retry(
    stop=stop_after_attempt(5),
    wait=wait_exponential(multiplier=1, min=1, max=30),
    retry=retry_if_exception_type((httpx.RequestError,)),
    reraise=True,
)
def fetch(client: httpx.Client, url: str) -> httpx.Response:
    r = client.get(url)
    if r.status_code in TRANSIENT_STATUS:
        raise httpx.RequestError(f"transient {r.status_code}", request=r.request)
    return r


## 7. Query parameters, the right way

Don't build URLs with string concatenation. It's error-prone (forgotten `?`, missing URL-encoding) and ugly. Pass `params` instead and let `httpx` do it.

In [ ]:
r = httpx.get(
    "https://httpbin.org/get",
    params={"q": "korean indie albums", "year": 2025, "page": 1},
)
print(r.url)            # see the encoded URL
print(r.json()["args"]) # httpbin echoes parsed query args


## 8. POST and forms

A POST sends data in the request body. There are three common body shapes:

- **Form-encoded** (`application/x-www-form-urlencoded`) — old-school HTML forms. Use the `data=` keyword.
- **JSON** (`application/json`) — modern APIs. Use the `json=` keyword.
- **Multipart** (`multipart/form-data`) — file uploads. Use the `files=` keyword.

Crawlers POST less often than they GET, but you'll need it for: logging in, submitting search forms, calling JSON APIs.

In [ ]:
# Form
r = httpx.post("https://httpbin.org/post", data={"username": "alice", "password": "wonderland"})
print("form ->", r.json()["form"])

# JSON
r = httpx.post("https://httpbin.org/post", json={"query": "python", "limit": 10})
print("json ->", r.json()["json"])


## 9. Redirects

By default `httpx.Client` does *not* follow redirects (unlike `requests`). This is a deliberate footgun-prevention design. Pass `follow_redirects=True` when you want the browser-like behavior.

In [ ]:
# Without redirect-following
r = httpx.get("https://httpbin.org/redirect/2")
print("no follow:", r.status_code, r.headers.get("location"))

# With redirect-following
with httpx.Client(follow_redirects=True) as c:
    r = c.get("https://httpbin.org/redirect/2")
    print("follow:   ", r.status_code, r.url, "-- history:", [h.url.path for h in r.history])


## 10. Putting it together — a polite fetch helper

By the end of this course you'll find yourself reaching for the same fetch helper over and over. Here's a starting version that bundles the lessons of this lecture.

In [ ]:
import httpx, time, random

DEFAULT_HEADERS = {
    "User-Agent": "CrawlingTutorial/0.1 (+contact@example.com)",
    "Accept-Language": "en-US,en;q=0.9",
}
TRANSIENT_STATUS = {429, 500, 502, 503, 504}


def make_client() -> httpx.Client:
    return httpx.Client(
        headers=DEFAULT_HEADERS,
        timeout=10.0,
        follow_redirects=True,
        http2=True,
    )


def polite_fetch(client: httpx.Client, url: str, max_attempts: int = 5) -> httpx.Response:
    last_error = None
    for attempt in range(max_attempts):
        try:
            r = client.get(url)
        except httpx.RequestError as e:
            last_error = e
        else:
            if r.status_code not in TRANSIENT_STATUS:
                return r
            last_error = httpx.HTTPStatusError(
                f"transient {r.status_code}", request=r.request, response=r
            )
        sleep_for = (2 ** attempt) + random.uniform(0, 1)
        time.sleep(sleep_for)
    raise last_error


# usage:
with make_client() as client:
    r = polite_fetch(client, "https://httpbin.org/get")
    print(r.status_code, r.json()["headers"]["User-Agent"])


## Recap

- Use a `Client` (or `Session`) — never a bare `httpx.get` in a loop.
- Set a real `User-Agent` that identifies your crawler.
- Always set a timeout. Always.
- Status codes 4xx and 5xx are real and need real handling. 429 and 5xx are retry-with-backoff. 4xx (other than 429) are usually permanent.
- Use `params=` for query strings, `data=`/`json=`/`files=` for POST bodies. No string concatenation.

## Exercises

1. Build a fetch helper like `polite_fetch` above, but make the `User-Agent` and `max_attempts` configurable. Add a `delay` parameter that sleeps a fixed amount between attempts (in addition to backoff).
2. Use `httpbin.org/status/{code}` to test your helper against 200, 404, 429, 503. Verify behavior matches your expectations.
3. Hit `https://httpbin.org/cookies/set?theme=dark&lang=ko` then `/cookies`. Without using a `Client`, do you see the cookies on the second call? With a `Client`?
4. Find a real (small, friendly) site that returns JSON. Fetch it with your helper. Print three top-level keys.

## Up next

**Lecture 03** — turning HTML bytes into queryable trees with BeautifulSoup. CSS selectors, XPath, when each shines, and how to write extractors that don't break the moment the site touches its template.
